# Aashish AI — Stage B training, instruction tuning (§7.4, P5)

Continues a Stage A checkpoint on the §7.4 instruction data with **assistant-only
loss**. It exists because Stage B had data and no trainer, and that mattered more
than it sounds: the runtime frame is essentially absent from the pretraining
corpus — of 17,265 seed documents, `<|ctx|>` appears in **4**, `<|asst|>` in
1,108, `<|sys|>` in 415. A Stage A-only checkpoint has therefore never been
shown the format it is asked to continue at inference time, and no amount of
extra pretraining steps on the same shards changes that.

**Status: the path is MEASURED at the local shape, the scale is NOT TESTED.**
`training/checkpoints/sft-local` holds a 60-step run from this laptop: loss
7.5571 → 6.3133 (windowed gate PASS), val 6.2715 → 5.5403, and the same prompts
through `inference/sample_answers.py` show the frame and `<|end|>` appearing
where Stage A emitted neither. Sixty steps on 3,000 examples is a wiring proof,
not quality. **No Kaggle session has run this notebook yet**, so treat the first
run as the test.

Read section 4 before spending GPU hours: the mask is the whole contribution of
Stage B, and it is printed rather than asserted.

## 1. Setup

In [ ]:
!nvidia-smi

import subprocess
import sys

import torch

print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('python', sys.version.split()[0], '| commit',
      subprocess.run(['git', 'rev-parse', '--short', 'HEAD'],
                     capture_output=True, text=True).stdout.strip())
print('bf16 supported:',
      torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)
# A T4 has no bf16 — which is why this notebook uses fp16 AMP, like Stage A.

In [ ]:
import os
import pathlib
import shutil
import tarfile

WORK = pathlib.Path('/kaggle/working')
MOUNT = pathlib.Path('/kaggle/input')
HOME = WORK / 'aashish-ai-portfolio'


def ensure_repo():
    """Where this session's repository is — creating it if it arrived as an archive.

    Four wrong guesses about this one mount were found the hard way, so this no
    longer guesses. A Kaggle Dataset does **not** mount at `/kaggle/input/<name>`;
    "Your Datasets" land under `/kaggle/input/datasets/<username>/<slug>/`. And
    Kaggle does **not** unpack a `.tgz` for you: the first real session found the
    repository sitting on the mount as a 1.2 MB file, with no `package.json`
    anywhere and the cell dying on `StopIteration`.

    So: two direct paths (a clone, a manual copy), then a search for a
    `package.json` beside `training/notebooks/`, then the archive. All offline,
    and none of it needs an edit the next time the layout changes.
    """
    for candidate in (HOME, WORK / 'newportfolio'):
        if (candidate / 'training').is_dir():
            return candidate
    for marker in MOUNT.rglob('package.json'):
        if (marker.parent / 'training' / 'notebooks').is_dir():
            return marker.parent
    for archive in sorted(list(MOUNT.rglob('*.tgz'))
                          + list(MOUNT.rglob('*.tar.gz'))):
        # `git archive ... HEAD` records members from the repository root, so the
        # members are the repository itself and there is no wrapping directory.
        HOME.mkdir(parents=True, exist_ok=True)
        with tarfile.open(archive) as tar:
            tar.extractall(HOME, filter='data')
        if (HOME / 'training').is_dir():
            return HOME
    return None


repo = ensure_repo()
if repo is None:
    raise SystemExit(
        f'No repository found. Looked in {HOME} and {WORK / "newportfolio"}, and '
        f'for a package.json or an archive under {MOUNT}. Attach the '
        f'aashish-ai-portfolio dataset, or clone the branch into {HOME}.')

# A Dataset mount is read-only, and this run writes data/sources.json (the
# licence verification itself), data/raw/, data/extracted/, data/processed/ and
# the checkpoints long before it trains anything: from the mount, the first
# write dies with OSError: [Errno 30] and the GPU is already allocated. So copy
# it out first, then prove the copy is writable rather than assuming it.
os.chdir(repo)
if pathlib.Path.cwd().is_relative_to(MOUNT):
    if not HOME.exists():
        shutil.copytree(pathlib.Path.cwd(), HOME)
    os.chdir(HOME)
pathlib.Path('.tmp-writable').write_text('ok')
pathlib.Path('.tmp-writable').unlink()

print('cwd:', os.getcwd())
TOKENIZER = 'ai/tokenizer/artifacts/stage-a-16k'
DATA = 'data/instruction/sft.jsonl'
RUN = '/kaggle/working/checkpoints/stage-b'

# The Stage A checkpoint is published as a dataset and *found* here, never named.
# The first version of this cell hard-coded /kaggle/working/checkpoints/stage-a,
# which is this session's own ephemeral directory: Stage A runs in a different
# kernel version and /kaggle/working is wiped when that session ends, so the path
# could never hold a checkpoint. It printed `stage A checkpoint: False` and
# carried on, so the real failure surfaced two cells later as a missing --init and
# read like a Stage B fault.
#
# Same lesson as the corpus cache, learned the same day: a Kaggle Dataset does
# not mount at /kaggle/input/<name>, it lands under datasets/<user>/<slug>/, so
# the mount is searched rather than guessed. The dataset holds one archive
# (stage-a-checkpoint.tgz) because a directory of files does not survive
# `kaggle datasets create` -- it uploaded one file out of 3,521 and reported
# success.
def find_stage_a():
    for marker in sorted(MOUNT.rglob('latest.pt')):
        if (marker.parent / 'RUN_MANIFEST.json').is_file():
            return marker.parent
    for archive in sorted(MOUNT.rglob('stage-a-checkpoint.tgz')):
        root = pathlib.Path(WORK) / 'stage-a-dl'
        root.mkdir(parents=True, exist_ok=True)
        with tarfile.open(archive) as tar:
            tar.extractall(root, filter='data')
        # the archive may or may not wrap its payload in a directory
        for candidate in [root, *sorted(p for p in root.iterdir() if p.is_dir())]:
            if (candidate / 'latest.pt').is_file() and \
                    (candidate / 'RUN_MANIFEST.json').is_file():
                return candidate
    return None


STAGE_A = find_stage_a()
if STAGE_A is None:
    raise SystemExit(
        'No Stage A checkpoint found. Looked for a RUN_MANIFEST.json beside a '
        'latest.pt, and for stage-a-checkpoint.tgz, under ' + str(MOUNT) + '. '
        'Stage B initialises from a Stage A checkpoint and cannot run without '
        'one; attach the dataset that carries it (or re-run Stage A in this same '
        'session, which is the only case where /kaggle/working holds one).')

print('stage A checkpoint:', STAGE_A)

# These two used to be printed, not checked. Measured 2026-10-03: the
# repository dataset is a `git archive` of HEAD, and
# `ai/tokenizer/artifacts/stage-a-16k` was gitignored as a build output, so no
# dataset a session can see carried it. Cell 3 printed `tokenizer: False` and
# carried on; cell 5 then died two cells later opening a `meta.json` that was
# never there, and the failure read as a Stage B fault rather than a missing
# Stage A artifact. A missing input is a reason to stop before the GPU is
# spent, which is the whole point of the cell.
if not pathlib.Path(TOKENIZER, 'tokenizer.json').is_file():
    raise SystemExit(
        f'No tokenizer at {TOKENIZER}. Stage B tokenises with the same '
        'vocabulary the Stage A checkpoint was trained on, and an --init from '
        'a different one loads some keys and trains the rest at random with '
        'no error to notice. The artifact is committed at '
        'ai/tokenizer/artifacts/stage-a-16k; if this says it is missing, the '
        'dataset attached to this kernel is older than that commit.')

print('tokenizer:', True)
# The instruction data does not exist yet - cell 7 builds it - so its absence
# is the expected state here and is reported rather than enforced.
print('instruction data:', pathlib.Path(DATA).is_file())

# `python -m` from here puts the repo root on sys.path, which is why every
# command below is a module rather than a path.


## 2. What Stage A left behind

Stage B continues a Stage A checkpoint. `--init` loads **model weights only**:
the optimizer, scaler and scheduler start fresh, because Stage B is a different
objective on different data and carrying Adam moments across is not "resuming".
The flag refuses a checkpoint whose vocab, width or depth differ from this run's
config — a checkpoint from another shape would otherwise load some keys and
leave the rest at their initial values, with no error to notice.

In [ ]:
!ls -la $STAGE_A

# Read the step count and the loss history here. Stage B's run manifest records
# what it was initialised from, so the two runs stay joined up on the record
# rather than in memory.
import json

manifest = json.load(open(f'{STAGE_A}/RUN_MANIFEST.json', encoding='utf-8'))
print('stage A step', manifest['hyperparameters']['steps'],
      '| tokenizer', manifest['tokenizer_version'],
      '| params', f"{manifest['params']:,}")
print('tokenizer must match this notebook:', manifest['tokenizer_version'] ==
      json.load(open(f'{TOKENIZER}/meta.json', encoding='utf-8'))['tokenizer_version'])

## 3. The instruction data (§7.4)

40,000 examples generated from `knowledge.json` — context-grounded, ≥30%
counterfactual contexts, 15% abstention, the §7.4 mix exact by largest-remainder
allocation. Nothing private is in it: the generator's fact index is the public
view only, because a model that learns a withheld value is one prompt away from
emitting it and no runtime filter can un-learn it.

In [ ]:
!python -m training.scripts.make_instruction_data --count 40000 --tokenizer $TOKENIZER

# The run prints the mix it achieved against the mix it asked for, the
# counterfactual share, and the token count MEASURED with the shipping
# tokenizer. Do not schedule from the ESTIMATED figure: on the 40k set it
# was 7,263,142 against a measured 6,846,206 — 6% high, not low, because
# this tokenizer is efficient on this text rather than clumsy at it.
#
# `--tokenizer $TOKENIZER` is load-bearing and was missing until 2026-10-03.
# The flag defaults to `ai/tokenizer/artifacts/seed-1k`, the 1k dev fixture,
# so this cell was reporting a count from a *different vocabulary* than the
# one Stage B trains with: 10,582,527 tokens measured with the 1k tokenizer
# against 6,846,206 with the real one, a 55% overstatement, and 1,155,200
# supervised against 801,703. The note above this cell used to quote the 1k
# figures as if they were the shipping tokenizer's. Only 11.7% of the stream
# is loss-bearing, so the supervised figure is the one that sizes a run.
!cat data/instruction/manifest.json

## 4. Look at the mask before spending GPU hours

Stage B's entire contribution is *which tokens the loss is on*. A wrong mask
does not crash and does not bend the loss curve in an obviously wrong way: the
run trains on the question, or silently drops the first word of every answer,
and the result is a model that writes plausible questions instead of answers.

This pass needs no torch and prints the mask for real examples.

In [ ]:
!python -m training.scripts.train_stage_b --pipeline-only --tokenizer $TOKENIZER --data $DATA --block 1024 --batch 8

# Read the `loss is on:` lines. They must be the assistant's answers plus their
# `<|end|>` tokens — no question, no rules, no `<|sys|>`/`<|ctx|>`/`<|user|>`.
# The pass also prints the supervised share of the stream (about 10.9%), which
# is the number to size a step budget from: an epoch is 10.58M tokens, of which
# only ~1.16M carry loss.
#
# It also checks the two things that would otherwise be found by a NaN or a
# silent no-op: every window handed to the model carries supervision (an
# all-ignored target makes cross_entropy return NaN and one NaN step poisons
# every weight in the run), and a saved cursor regenerates the identical next
# batch.

## 5. Persistence first

Kaggle sessions are ephemeral. Push or save the run directory before training,
not after — a session that ends without either loses everything.

In [ ]:
#  - push /kaggle/working/checkpoints/stage-b to a Kaggle Dataset / Drive / HF Hub, or
#  - write to /kaggle/working and Save Version every session.
!mkdir -p $RUN
!ls -la $RUN

## 6. Stage B training

`--max-minutes` stops cleanly and writes a checkpoint before the time box ends,
so an interrupted run is a resume rather than a loss. The learning rate is
lower than Stage A's on purpose: starting an SFT run at the pretraining rate
undoes the pretraining.

In [ ]:
!python -m training.scripts.train_stage_b --config A --tokenizer $TOKENIZER --data $DATA --init $STAGE_A/latest.pt --run-dir $RUN --steps 500 --batch 8 --block 1024 --grad-accum 2 --lr 5e-5 --warmup 100 --eval-every 100 --save-every 100 --amp --gate --max-minutes 540 --resume auto

# 8 x 1024 x 2 is ~16K tokens per optimizer step, of which 11.2% carry
# loss. The gate is the windowed one: first five logged steps against the
# last five, so a run that ends slightly above its minimum cannot fail it.
#
# `--steps 500`, down from 3000. MEASURED on v1: val loss was 0.0182 at
# step 500 (0.7 epochs), 0.0151 at step 900, and 0.0083 at step 3000. About
# one epoch was worth having; the remaining six epochs bought under 0.01 of
# val loss and produced the template-reciting behaviour. On the current
# measured stream of 7,158,788 tokens one epoch is ~437 steps, so 500 is
# ~1.14 epochs. Held-out loss is a weak generalisation measure on templated
# data either way - cell 17 samples novel prompts for the real check.


## 7. Every subsequent session

The identical command. It resumes from `latest.pt` and reports the step, the
tokens and supervised tokens already consumed, and the loss history.

In [ ]:
!ls -la $RUN
!cat $RUN/RUN_MANIFEST.json

# A resume that silently began a new run would look like success here, so check
# that the step continues from where the previous session stopped, and that the
# `tokens_seen_cumulative` figure is not zero. A zeroed cursor reads exactly
# like a lost data cursor — that bug existed and was found by running this.

## 8. See the answers, not just the loss

A low loss and a useful answer are different claims. This samples the runtime
frame — the prompt the browser builds, cut at `<|asst|>` — through the same
numpy forward the JS engine is verified against, and prints both checkpoints
side by side. It reports the prompt echo rate too, because an answer that
copies its input scores well and is useless.

In [ ]:
!python inference/sample_answers.py --checkpoint stage-A=$STAGE_A --checkpoint stage-B=$RUN --n 4 --tokens 32

# What to look for, in order:
#   1. does the answer end with `<|end|>` (does the model know a turn ends)?
#   2. does it answer, or does it echo the context back?
#   3. does it abstain with `<|abstain|>` when the context has no answer?
# A run that has learnt (1) but not (2) is a run that needs more of the same
# data, not different hyperparameters.
# ### 8b. The same eight novel prompts that showed v1 failing
#
# Cell 17 above samples four prompts *drawn from the training set*, which is
# why v1 could pass it while answering three unrelated hand-written questions
# with one byte-identical reply. These eight were written by hand and are in
# the same jsonl shape, so `--data` reads them with no code change.
#
# v1 on this exact set: **8 prompts → 5 distinct answers**, the commonest
# byte-identical across three unrelated questions, and 3 of 8 leaking raw
# `<|fact:...|>` slots. The number to beat is 5.
!python inference/sample_answers.py --checkpoint stage-A=$STAGE_A --checkpoint stage-B=$RUN --data evaluation/novel-prompts.jsonl --n 8 --tokens 48


## 9. Export (P6)

`npm run export:model -- --run-dir <this run>` then `npm run verify:engine`,
which is the gate that compares the JavaScript engine against the numpy
reference **on the shipping weights**. Export is not scaffolded further here on
purpose: it needs a finished checkpoint, and a pipeline built against a
checkpoint that does not exist yet is a pipeline whose interface is invented.